### Datos Electorales

In [8]:
import pandas as pd

# ----- Function Definitions -----
def harmonize_agrupacion_id(agrupacion_id):
    if pd.isna(agrupacion_id):
        return "000000"
    else:
        try:
            return str(int(float(agrupacion_id))).zfill(6)
        except ValueError:
            return agrupacion_id

# ----- Data Loading -----
df19 = pd.read_csv('./../datos/BD/votos_eleccion_14_table.csv')
df23 = pd.read_csv('./../datos/BD/votos_eleccion_17_table.csv')
df = pd.concat([df19, df23], axis=0)
df = df.loc[(df.cargo_id.isin([1, 4, 7])) & (df.distrito_id == 2)]

cargo = pd.read_csv('./../datos/BD/cargo_tags.csv')
agrup_lista = pd.read_csv('./../datos/BD/agrupacion_lista_table.csv')
claves_dptos = pd.read_csv('./../datos/BD/claves_dptos_ref.csv')
eleccion_tags = pd.read_csv('./../datos/BD/eleccion_tags.csv')
prov_nams = pd.read_csv('./../datos/BD/distrito_table.csv')

# Agrup Lista Data Preprocessing
agrup_lista['agrupacion_id'] = agrup_lista['agrupacion_id'].apply(harmonize_agrupacion_id)
agrup_lista['lista_numero'] = agrup_lista['lista_numero'].apply(harmonize_agrupacion_id)



/tmp/ipykernel_662694/4151758115.py:14: DtypeWarning: Columns (6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df19 = pd.read_csv('./../datos/BD/votos_eleccion_14_table.csv')
/tmp/ipykernel_662694/4151758115.py:15: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df23 = pd.read_csv('./../datos/BD/votos_eleccion_17_table.csv')


In [9]:
import pandas as pd

def harmonize_agrupacion_names(df, col_name='agrupacion_nombre'):
    """Harmonizes the names of agrupaciones."""
    replacements = {
        'CAMBIEMOS BUENOS AIRES': 'CAMBIEMOS',
        'JUNTOS': 'JUNTOS POR EL CAMBIO'
    }
    df[col_name] = df[col_name].replace(replacements).str.title().str.strip()
    return df


In [40]:
import pandas as pd

# ----- Data Merging & Transformations -----
# Merge with simil_nombre
simil_nombre = agrup_lista.groupby(['eleccion_id', 'distrito_id', 'agrupacion_id']).agrupacion_nombre.first().reset_index()
df['agrupacion_id'] = df['agrupacion_id'].astype(str).str.zfill(6)
merged_data = df.merge(simil_nombre)
merged_data = harmonize_agrupacion_names(merged_data)
# display(merged_data)

# Group and aggregate data
aggregated_data = merged_data.groupby(['eleccion_id', 'cargo_id', 'agrupacion_nombre', 'lista_numero', 'votos_tipo']).agg({'votos_cantidad': 'sum'}).reset_index()
# display(aggregated_data)

# Top N aggregation
N = 50
top_aggregated_data = aggregated_data.groupby(['eleccion_id', 'cargo_id', 'votos_tipo'])\
                                     .apply(lambda x: x.nlargest(N, 'votos_cantidad'))\
                                     .reset_index(drop=True).rename(columns={'votos_cantidad': 'votos_nacional'})
# display(top_aggregated_data)

# Further transformations on original data
data_copy = df.copy()
data_copy = harmonize_agrupacion_names(data_copy)
data_copy = data_copy.merge(top_aggregated_data, how='left')
data_copy['agrupacion_nombre_'] = data_copy['agrupacion_nombre'].mask(data_copy['votos_nacional'].isnull(), 'Resto')

data_aggregated = data_copy.groupby(['distrito_id', 'seccion_id', 'circuito_id', 'mesa_id', 'cargo_id', 'agrupacion_nombre_', 'lista_numero', 'votos_tipo', 'eleccion_id'])\
                           .agg({'votos_cantidad': 'sum'}).reset_index()


# More transformations...
data_circ = data_aggregated.groupby(['eleccion_id', 'cargo_id', 'agrupacion_nombre_', 'lista_numero', 'votos_tipo', 'distrito_id', 'seccion_id', 'circuito_id'])[['votos_cantidad']].sum()
data_circ = data_circ.reset_index()
data_circ = data_circ.merge(eleccion_tags).merge(cargo)


# Group by 'eleccion_tag', 'cargo_tag', and 'in1_prov', and calculate the sum of 'votos_cantidad', divide for PCT
sum_votes = data_circ.groupby(['eleccion_tag', 'cargo_tag', 'distrito_id', 'seccion_id', 'votos_tipo', 'circuito_id'])['votos_cantidad'].transform('sum')
data_circ['votos_porcentaje'] = data_circ['votos_cantidad'] / sum_votes
votos_agrup_lista_circ = data_circ.reset_index(drop = True)

# votos_agrup_lista_circ.to_csv('./../datos/out/votos_agrup_lista_circ.csv', index = False)


# votos_agrup_lista_circ = pd.read_csv('./../datos/out/votos_agrup_lista_circ.csv')



In [41]:

# --- Load Data ---
print("Loading data...")

# Load necessary datasets
radio_region = pd.read_csv('./../datos/info/radio_ref.csv', usecols=['radio', 'NOMDPTO', 'Region'])
radios_circuitos_secciones = pd.read_csv('./../datos/info/radios_circuitos_secciones_ref.csv')[['COD_2010', 'distrito_id', 'seccion_id', 'seccion_nombre']]
prov_nams = pd.read_csv('./../datos/BD/distrito_table.csv')

# --- Data Preprocessing ---
print("Processing data...")

# Radio region processing
radio_region['COD_2010'] = radio_region['radio'].astype(str).str.zfill(9)

# Merge data
merge_data = radios_circuitos_secciones.merge(radio_region, on='COD_2010', how='left')
seccion_region = merge_data.drop(['COD_2010', 'radio'], axis=1).drop_duplicates()
seccion_region = seccion_region.groupby(['distrito_id', 'seccion_id', 'seccion_nombre']).first().reset_index()

# Votos agrup processing
data_circ_ix = votos_agrup_lista_circ.set_index(['distrito_id', 'seccion_id', 'circuito_id', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_', 'lista_numero', 'votos_tipo'])
votos_circuito = data_circ_ix['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_', 'lista_numero'])['PASO23n']['PR'].sum(1).sort_values(ascending=False)
circuitos_ppales = votos_circuito[votos_circuito > 1000].index.to_frame().reset_index(drop=True)

# Lista processing
votos_lista = votos_agrup_lista_circ.groupby(['eleccion_tag', 'votos_tipo', 'agrupacion_nombre_', 'lista_numero'])['votos_cantidad'].sum().sort_values(ascending=False)

N = 50
main_listas = votos_lista.head(N).index.to_frame().reset_index(drop=True)
display(main_listas)

# --- Merge Information ---
print("Merging data...")

info = circuitos_ppales.merge(votos_agrup_lista_circ)
info = main_listas.merge(info)
info = info.merge(prov_nams).merge(seccion_region, how='left')

info.head()


Loading data...
Processing data...


,eleccion_tag,votos_tipo,agrupacion_nombre_,lista_numero
0,PASO19n,POSITIVO,Frente De Todos,2
1,PASO19n,POSITIVO,Juntos Por El Cambio,1A
2,PASO23n,POSITIVO,Union Por La Patria,3604
3,PASO19n,POSITIVO,Frente De Todos,A
4,PASO23n,POSITIVO,La Libertad Avanza,3612
5,PASO19n,POSITIVO,Juntos Por El Cambio,A
6,PASO23n,POSITIVO,Juntos Por El Cambio,3603
7,PASO23n,POSITIVO,Juntos Por El Cambio,3602
8,PASO23n,BLANCO,Resto,0
9,PASO23n,POSITIVO,La Libertad Avanza,3016


Merging data...


,eleccion_tag,votos_tipo,agrupacion_nombre_,lista_numero,distrito_id,seccion_id,circuito_id,eleccion_id,cargo_id,votos_cantidad,...,eleccion_tipo,recuento_tipo,padron_tipo,cargo_nombre,cargo_tag,votos_porcentaje,distrito_nombre,seccion_nombre,NOMDPTO,Region
0,PASO19n,POSITIVO,Frente De Todos,2,2,77,00652A,14,4,63821,...,PASO,PROVISORIO,NORMAL,Gobernador,GB,0.687837,Buenos Aires,Merlo,Merlo,Gran Buenos Aires
1,PASO19n,POSITIVO,Frente De Todos,2,2,77,00652A,14,7,50279,...,PASO,PROVISORIO,NORMAL,Intendente,IN,0.547946,Buenos Aires,Merlo,Merlo,Gran Buenos Aires
2,PASO19n,POSITIVO,Frente De Todos,2,2,61,00635B,14,4,58637,...,PASO,PROVISORIO,NORMAL,Gobernador,GB,0.733100,Buenos Aires,La Matanza,La Matanza,Gran Buenos Aires
3,PASO19n,POSITIVO,Frente De Todos,2,2,61,00635B,14,7,57783,...,PASO,PROVISORIO,NORMAL,Intendente,IN,0.728893,Buenos Aires,La Matanza,La Matanza,Gran Buenos Aires
4,PASO19n,POSITIVO,Frente De Todos,2,2,80,000665,14,4,38462,...,PASO,PROVISORIO,NORMAL,Gobernador,GB,0.519385,Buenos Aires,Morón,Morón,Gran Buenos Aires


In [87]:
x = info.groupby(['seccion_id'])['votos_cantidad'].sum().sort_values(ascending = False)#
main_partidos = x.loc[x > 3e5].index.values

# x = x.reset_index(drop = True)
# (x.cumsum()/x.sum()).plot()
# x.head(40)

In [88]:
info = info.loc[info.seccion_id.isin(main_partidos)]
tabla = info.loc[info.cargo_tag == 'IN'].groupby(['seccion_id', 'seccion_nombre', 'Region', 'agrupacion_nombre_', 'eleccion_tag'])['votos_cantidad'].sum().reset_index()
tabla
total_votos_seccion = tabla.groupby(['seccion_id', 'eleccion_tag'])['votos_cantidad'].sum().reset_index()
tabla = tabla.merge(total_votos_seccion, on=['seccion_id', 'eleccion_tag'], suffixes=('', '_total'))
tabla['votos_porcentaje'] = (tabla['votos_cantidad'] / tabla['votos_cantidad_total']) * 100

idx_max = tabla.groupby(['seccion_id', 'eleccion_tag'])['votos_cantidad'].idxmax()

mayores_agrupaciones = tabla.loc[idx_max].sort_values(by=['Region', 'seccion_id', 'eleccion_tag'])


In [89]:
from IPython.display import display, Markdown

# Función para mostrar la tabla de votos y porcentajes por elección
def mostrar_tabla(data, eleccion):
    df = data[data['eleccion_tag'] == eleccion]
    df = df.sort_values(by='votos_cantidad', ascending=False)
    table_md = "| Agrupación | Votos | Porcentaje |\n|---|---|---|\n"
    for _, row in df.iterrows():
        table_md += f"| {row['agrupacion_nombre_']} | {row['votos_cantidad']} | {row['votos_porcentaje']:.2f}% |\n"
    return table_md

# Presentar la información
output_str = ""

for region, data_region in mayores_agrupaciones.groupby('Region'):
    output_str += f"\n## Región: {region}\n"
    for seccion, data_seccion in data_region.groupby('seccion_id'):
        row = data_seccion.iloc[0]
        output_str += f"\n### Sección: {row['seccion_nombre']}\n"
        
        # Tabla 2019
        output_str += "\n**Elección 2019**\n"
        output_str += mostrar_tabla(tabla[tabla['seccion_id'] == seccion], 'PASO19n')
        
        # Tabla 2023
        output_str += "\n**Elección 2023**\n"
        output_str += mostrar_tabla(tabla[tabla['seccion_id'] == seccion], 'PASO23n')

display(Markdown(output_str))



## Región: Gran Buenos Aires

### Sección: Almirante Brown

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 188633 | 62.28% |
| Juntos Por El Cambio | 72967 | 24.09% |
| Consenso Federal | 19969 | 6.59% |
| Frente De Izquierda Y De Trabajadores - Unidad | 13586 | 4.49% |
| Frente Nos | 3626 | 1.20% |
| Movimiento De Avanzada Socialista | 3148 | 1.04% |
| Frente Patriota | 931 | 0.31% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 118118 | 40.37% |
| La Libertad Avanza | 66617 | 22.77% |
| Juntos Por El Cambio | 52926 | 18.09% |
| Resto | 35435 | 12.11% |
| Frente De Izquierda Y De Trabajadores - Unidad | 12364 | 4.23% |
| Principios Y Valores | 2651 | 0.91% |
| Movimiento Libres Del Sur | 1484 | 0.51% |
| Movimiento De Avanzada Socialista | 1406 | 0.48% |
| Frente Patriota Federal | 856 | 0.29% |
| Politica Obrera | 750 | 0.26% |

### Sección: Avellaneda

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 122160 | 59.85% |
| Juntos Por El Cambio | 62033 | 30.39% |
| Consenso Federal | 11401 | 5.59% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6475 | 3.17% |
| Movimiento De Avanzada Socialista | 1517 | 0.74% |
| Frente Patriota | 540 | 0.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 89113 | 43.50% |
| Juntos Por El Cambio | 51812 | 25.29% |
| La Libertad Avanza | 33143 | 16.18% |
| Resto | 20839 | 10.17% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6531 | 3.19% |
| Principios Y Valores | 1495 | 0.73% |
| Movimiento De Avanzada Socialista | 694 | 0.34% |
| Movimiento Libres Del Sur | 481 | 0.23% |
| Frente Patriota Federal | 391 | 0.19% |
| Politica Obrera | 370 | 0.18% |

### Sección: Berazategui

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 123847 | 67.95% |
| Juntos Por El Cambio | 38875 | 21.33% |
| Consenso Federal | 8342 | 4.58% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5925 | 3.25% |
| Frente Nos | 3489 | 1.91% |
| Movimiento De Avanzada Socialista | 1286 | 0.71% |
| Frente Patriota | 494 | 0.27% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 81885 | 42.86% |
| La Libertad Avanza | 38825 | 20.32% |
| Juntos Por El Cambio | 37514 | 19.64% |
| Resto | 22559 | 11.81% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6513 | 3.41% |
| Principios Y Valores | 1870 | 0.98% |
| Movimiento De Avanzada Socialista | 717 | 0.38% |
| Movimiento Libres Del Sur | 667 | 0.35% |
| Politica Obrera | 496 | 0.26% |

### Sección: Esteban Echeverría

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 96614 | 59.15% |
| Juntos Por El Cambio | 44889 | 27.48% |
| Consenso Federal | 11380 | 6.97% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5985 | 3.66% |
| Frente Nos | 2622 | 1.61% |
| Movimiento De Avanzada Socialista | 1265 | 0.77% |
| Frente Patriota | 571 | 0.35% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 59071 | 34.81% |
| La Libertad Avanza | 40140 | 23.65% |
| Juntos Por El Cambio | 39937 | 23.53% |
| Resto | 20361 | 12.00% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6258 | 3.69% |
| Principios Y Valores | 1299 | 0.77% |
| Movimiento Libres Del Sur | 1061 | 0.63% |
| Movimiento De Avanzada Socialista | 652 | 0.38% |
| Frente Patriota Federal | 520 | 0.31% |
| Politica Obrera | 420 | 0.25% |

### Sección: Florencio Varela

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 103294 | 52.19% |
| Juntos Por El Cambio | 55011 | 27.79% |
| Consenso Federal | 26496 | 13.39% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7149 | 3.61% |
| Frente Nos | 3478 | 1.76% |
| Movimiento De Avanzada Socialista | 1669 | 0.84% |
| Frente Patriota | 831 | 0.42% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 82562 | 38.21% |
| La Libertad Avanza | 48447 | 22.42% |
| Juntos Por El Cambio | 38615 | 17.87% |
| Resto | 32548 | 15.06% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8454 | 3.91% |
| Principios Y Valores | 2128 | 0.98% |
| Movimiento Libres Del Sur | 1143 | 0.53% |
| Movimiento De Avanzada Socialista | 942 | 0.44% |
| Frente Patriota Federal | 652 | 0.30% |
| Politica Obrera | 567 | 0.26% |

### Sección: General San Martín

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 125320 | 55.35% |
| Juntos Por El Cambio | 70243 | 31.03% |
| Consenso Federal | 14972 | 6.61% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9500 | 4.20% |
| Frente Nos | 3672 | 1.62% |
| Movimiento De Avanzada Socialista | 1938 | 0.86% |
| Frente Patriota | 761 | 0.34% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 88680 | 37.59% |
| Juntos Por El Cambio | 66607 | 28.23% |
| La Libertad Avanza | 47941 | 20.32% |
| Resto | 21108 | 8.95% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7209 | 3.06% |
| Principios Y Valores | 1834 | 0.78% |
| Movimiento Libres Del Sur | 1117 | 0.47% |
| Movimiento De Avanzada Socialista | 889 | 0.38% |
| Politica Obrera | 538 | 0.23% |

### Sección: San Miguel

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 80257 | 51.07% |
| Frente De Todos | 63979 | 40.71% |
| Consenso Federal | 6817 | 4.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4912 | 3.13% |
| Movimiento De Avanzada Socialista | 1186 | 0.75% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 76298 | 44.88% |
| Union Por La Patria | 44659 | 26.27% |
| La Libertad Avanza | 25818 | 15.19% |
| Resto | 16466 | 9.69% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4681 | 2.75% |
| Principios Y Valores | 684 | 0.40% |
| Movimiento De Avanzada Socialista | 507 | 0.30% |
| Movimiento Libres Del Sur | 356 | 0.21% |
| Politica Obrera | 300 | 0.18% |
| Frente Patriota Federal | 244 | 0.14% |

### Sección: La Matanza

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 472575 | 64.65% |
| Juntos Por El Cambio | 160529 | 21.96% |
| Consenso Federal | 45946 | 6.29% |
| Frente De Izquierda Y De Trabajadores - Unidad | 30538 | 4.18% |
| Frente Nos | 13982 | 1.91% |
| Movimiento De Avanzada Socialista | 5195 | 0.71% |
| Frente Patriota | 2155 | 0.29% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 280043 | 39.77% |
| La Libertad Avanza | 148003 | 21.02% |
| Juntos Por El Cambio | 146442 | 20.80% |
| Resto | 82047 | 11.65% |
| Frente De Izquierda Y De Trabajadores - Unidad | 32213 | 4.57% |
| Principios Y Valores | 6091 | 0.86% |
| Movimiento Libres Del Sur | 3405 | 0.48% |
| Movimiento De Avanzada Socialista | 2334 | 0.33% |
| Frente Patriota Federal | 1808 | 0.26% |
| Politica Obrera | 1802 | 0.26% |

### Sección: Lanús

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 137462 | 51.12% |
| Juntos Por El Cambio | 98945 | 36.80% |
| Consenso Federal | 17052 | 6.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 10118 | 3.76% |
| Frente Nos | 3089 | 1.15% |
| Movimiento De Avanzada Socialista | 2218 | 0.82% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 95965 | 36.81% |
| Juntos Por El Cambio | 83133 | 31.89% |
| La Libertad Avanza | 45298 | 17.38% |
| Resto | 22063 | 8.46% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9439 | 3.62% |
| Principios Y Valores | 1949 | 0.75% |
| Movimiento Libres Del Sur | 1100 | 0.42% |
| Movimiento De Avanzada Socialista | 1040 | 0.40% |
| Politica Obrera | 683 | 0.26% |

### Sección: Lomas De Zamora

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 210394 | 61.04% |
| Juntos Por El Cambio | 91220 | 26.46% |
| Consenso Federal | 21552 | 6.25% |
| Frente De Izquierda Y De Trabajadores - Unidad | 13839 | 4.01% |
| Frente Nos | 4422 | 1.28% |
| Movimiento De Avanzada Socialista | 3261 | 0.95% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 138101 | 38.01% |
| Juntos Por El Cambio | 93174 | 25.65% |
| La Libertad Avanza | 73391 | 20.20% |
| Resto | 37637 | 10.36% |
| Frente De Izquierda Y De Trabajadores - Unidad | 13398 | 3.69% |
| Principios Y Valores | 2881 | 0.79% |
| Movimiento Libres Del Sur | 1620 | 0.45% |
| Movimiento De Avanzada Socialista | 1522 | 0.42% |
| Frente Patriota Federal | 795 | 0.22% |
| Politica Obrera | 770 | 0.21% |

### Sección: Merlo

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 176748 | 77.70% |
| Juntos Por El Cambio | 23955 | 10.53% |
| Frente De Izquierda Y De Trabajadores - Unidad | 11131 | 4.89% |
| Consenso Federal | 10392 | 4.57% |
| Frente Nos | 2602 | 1.14% |
| Movimiento De Avanzada Socialista | 2048 | 0.90% |
| Frente Patriota | 591 | 0.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 73999 | 31.10% |
| La Libertad Avanza | 55527 | 23.34% |
| Juntos Por El Cambio | 52455 | 22.05% |
| Resto | 34653 | 14.56% |
| Frente De Izquierda Y De Trabajadores - Unidad | 12847 | 5.40% |
| Movimiento Libres Del Sur | 2890 | 1.21% |
| Principios Y Valores | 2251 | 0.95% |
| Movimiento De Avanzada Socialista | 1351 | 0.57% |
| Politica Obrera | 980 | 0.41% |
| Frente Patriota Federal | 969 | 0.41% |

### Sección: Moreno

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 155607 | 68.63% |
| Juntos Por El Cambio | 49095 | 21.65% |
| Consenso Federal | 10067 | 4.44% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7180 | 3.17% |
| Frente Nos | 2606 | 1.15% |
| Movimiento De Avanzada Socialista | 1513 | 0.67% |
| Frente Patriota | 657 | 0.29% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 113463 | 48.08% |
| La Libertad Avanza | 49823 | 21.11% |
| Juntos Por El Cambio | 35437 | 15.02% |
| Resto | 24633 | 10.44% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8115 | 3.44% |
| Movimiento Libres Del Sur | 1378 | 0.58% |
| Principios Y Valores | 1122 | 0.48% |
| Movimiento De Avanzada Socialista | 804 | 0.34% |
| Frente Patriota Federal | 621 | 0.26% |
| Politica Obrera | 582 | 0.25% |

### Sección: Morón

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 91484 | 46.74% |
| Juntos Por El Cambio | 75401 | 38.53% |
| Consenso Federal | 15097 | 7.71% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8326 | 4.25% |
| Frente Nos | 3488 | 1.78% |
| Movimiento De Avanzada Socialista | 1921 | 0.98% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 65435 | 33.63% |
| Union Por La Patria | 61399 | 31.55% |
| La Libertad Avanza | 39405 | 20.25% |
| Resto | 16370 | 8.41% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7687 | 3.95% |
| Principios Y Valores | 1976 | 1.02% |
| Movimiento De Avanzada Socialista | 816 | 0.42% |
| Movimiento Libres Del Sur | 688 | 0.35% |
| Politica Obrera | 421 | 0.22% |
| Frente Patriota Federal | 403 | 0.21% |

### Sección: Quilmes

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 175769 | 55.24% |
| Juntos Por El Cambio | 104654 | 32.89% |
| Consenso Federal | 21419 | 6.73% |
| Frente De Izquierda Y De Trabajadores - Unidad | 10240 | 3.22% |
| Frente Nos | 4090 | 1.29% |
| Movimiento De Avanzada Socialista | 1994 | 0.63% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 121126 | 37.00% |
| Juntos Por El Cambio | 103699 | 31.68% |
| La Libertad Avanza | 52858 | 16.15% |
| Resto | 34425 | 10.52% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9910 | 3.03% |
| Principios Y Valores | 1828 | 0.56% |
| Movimiento Libres Del Sur | 1210 | 0.37% |
| Movimiento De Avanzada Socialista | 1044 | 0.32% |
| Politica Obrera | 638 | 0.19% |
| Frente Patriota Federal | 631 | 0.19% |

### Sección: San Fernando

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 57377 | 64.50% |
| Juntos Por El Cambio | 23536 | 26.46% |
| Consenso Federal | 4761 | 5.35% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2654 | 2.98% |
| Movimiento De Avanzada Socialista | 634 | 0.71% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 46063 | 48.58% |
| Juntos Por El Cambio | 20951 | 22.10% |
| La Libertad Avanza | 14618 | 15.42% |
| Resto | 9763 | 10.30% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1910 | 2.01% |
| Movimiento Libres Del Sur | 657 | 0.69% |
| Principios Y Valores | 444 | 0.47% |
| Movimiento De Avanzada Socialista | 249 | 0.26% |
| Politica Obrera | 163 | 0.17% |

### Sección: San Isidro

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 83601 | 55.24% |
| Frente De Todos | 52864 | 34.93% |
| Consenso Federal | 7946 | 5.25% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5642 | 3.73% |
| Movimiento De Avanzada Socialista | 1281 | 0.85% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 106459 | 54.98% |
| Union Por La Patria | 32291 | 16.68% |
| La Libertad Avanza | 30966 | 15.99% |
| Resto | 15766 | 8.14% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4933 | 2.55% |
| Principios Y Valores | 1301 | 0.67% |
| Movimiento Libres Del Sur | 911 | 0.47% |
| Movimiento De Avanzada Socialista | 592 | 0.31% |
| Politica Obrera | 396 | 0.20% |

### Sección: Tigre

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 121630 | 56.39% |
| Juntos Por El Cambio | 67829 | 31.45% |
| Consenso Federal | 11534 | 5.35% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8413 | 3.90% |
| Frente Nos | 4731 | 2.19% |
| Movimiento De Avanzada Socialista | 1545 | 0.72% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 103033 | 45.28% |
| Juntos Por El Cambio | 52360 | 23.01% |
| La Libertad Avanza | 40177 | 17.66% |
| Resto | 22932 | 10.08% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6132 | 2.69% |
| Movimiento Libres Del Sur | 1061 | 0.47% |
| Principios Y Valores | 830 | 0.36% |
| Movimiento De Avanzada Socialista | 637 | 0.28% |
| Politica Obrera | 399 | 0.18% |

### Sección: Tres De Febrero

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 97576 | 49.43% |
| Juntos Por El Cambio | 72756 | 36.86% |
| Consenso Federal | 14437 | 7.31% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8254 | 4.18% |
| Frente Nos | 2614 | 1.32% |
| Movimiento De Avanzada Socialista | 1756 | 0.89% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 80356 | 40.85% |
| Union Por La Patria | 54591 | 27.75% |
| La Libertad Avanza | 33161 | 16.86% |
| Resto | 18539 | 9.43% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6574 | 3.34% |
| Principios Y Valores | 1638 | 0.83% |
| Movimiento Libres Del Sur | 749 | 0.38% |
| Movimiento De Avanzada Socialista | 670 | 0.34% |
| Politica Obrera | 419 | 0.21% |

### Sección: Vicente López

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 94223 | 57.14% |
| Frente De Todos | 44810 | 27.17% |
| Consenso Federal | 13707 | 8.31% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7467 | 4.53% |
| Frente Nos | 2876 | 1.74% |
| Movimiento De Avanzada Socialista | 1813 | 1.10% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 74038 | 47.62% |
| Union Por La Patria | 32107 | 20.65% |
| La Libertad Avanza | 27419 | 17.63% |
| Resto | 13085 | 8.42% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5994 | 3.85% |
| Principios Y Valores | 1214 | 0.78% |
| Movimiento De Avanzada Socialista | 653 | 0.42% |
| Movimiento Libres Del Sur | 436 | 0.28% |
| Politica Obrera | 289 | 0.19% |
| Frente Patriota Federal | 255 | 0.16% |

### Sección: J. C. Paz

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 88667 | 61.88% |
| Juntos Por El Cambio | 34687 | 24.21% |
| Consenso Federal | 9081 | 6.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6309 | 4.40% |
| Frente Nos | 3242 | 2.26% |
| Movimiento De Avanzada Socialista | 1305 | 0.91% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 51254 | 35.75% |
| La Libertad Avanza | 32913 | 22.95% |
| Juntos Por El Cambio | 29381 | 20.49% |
| Resto | 20609 | 14.37% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6025 | 4.20% |
| Principios Y Valores | 1049 | 0.73% |
| Movimiento Libres Del Sur | 679 | 0.47% |
| Movimiento De Avanzada Socialista | 642 | 0.45% |
| Frente Patriota Federal | 435 | 0.30% |
| Politica Obrera | 396 | 0.28% |

### Sección: Malvinas Argentinas

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 122679 | 64.31% |
| Juntos Por El Cambio | 50029 | 26.22% |
| Consenso Federal | 8201 | 4.30% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5460 | 2.86% |
| Frente Nos | 3292 | 1.73% |
| Movimiento De Avanzada Socialista | 1115 | 0.58% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 85884 | 45.03% |
| La Libertad Avanza | 38917 | 20.41% |
| Juntos Por El Cambio | 33974 | 17.81% |
| Resto | 21324 | 11.18% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6169 | 3.23% |
| Movimiento Libres Del Sur | 1483 | 0.78% |
| Principios Y Valores | 1274 | 0.67% |
| Movimiento De Avanzada Socialista | 738 | 0.39% |
| Frente Patriota Federal | 505 | 0.26% |
| Politica Obrera | 446 | 0.23% |

### Sección: Ezeiza

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 57314 | 68.47% |
| Juntos Por El Cambio | 19084 | 22.80% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3245 | 3.88% |
| Consenso Federal | 2563 | 3.06% |
| Frente Nos | 968 | 1.16% |
| Movimiento De Avanzada Socialista | 528 | 0.63% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 45556 | 46.44% |
| La Libertad Avanza | 20006 | 20.39% |
| Juntos Por El Cambio | 15705 | 16.01% |
| Resto | 12405 | 12.65% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2643 | 2.69% |
| Principios Y Valores | 528 | 0.54% |
| Frente Patriota Federal | 374 | 0.38% |
| Movimiento Libres Del Sur | 364 | 0.37% |
| Movimiento De Avanzada Socialista | 274 | 0.28% |
| Politica Obrera | 245 | 0.25% |

### Sección: Ituzaingo

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 46493 | 48.18% |
| Juntos Por El Cambio | 33763 | 34.99% |
| Consenso Federal | 9080 | 9.41% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4529 | 4.69% |
| Frente Nos | 1604 | 1.66% |
| Movimiento De Avanzada Socialista | 1020 | 1.06% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 33836 | 32.32% |
| Union Por La Patria | 33360 | 31.87% |
| La Libertad Avanza | 20384 | 19.47% |
| Resto | 10048 | 9.60% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4537 | 4.33% |
| Principios Y Valores | 907 | 0.87% |
| Movimiento De Avanzada Socialista | 543 | 0.52% |
| Movimiento Libres Del Sur | 401 | 0.38% |
| Frente Patriota Federal | 374 | 0.36% |
| Politica Obrera | 295 | 0.28% |

### Sección: Hurlingham

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 59719 | 55.18% |
| Juntos Por El Cambio | 32357 | 29.90% |
| Consenso Federal | 7853 | 7.26% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4560 | 4.21% |
| Frente Nos | 2824 | 2.61% |
| Movimiento De Avanzada Socialista | 908 | 0.84% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 53577 | 48.75% |
| Juntos Por El Cambio | 25598 | 23.29% |
| La Libertad Avanza | 17148 | 15.60% |
| Resto | 8972 | 8.16% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2852 | 2.60% |
| Principios Y Valores | 611 | 0.56% |
| Movimiento Libres Del Sur | 468 | 0.43% |
| Movimiento De Avanzada Socialista | 336 | 0.31% |
| Politica Obrera | 333 | 0.30% |

## Región: Pampeana

### Sección: Bahía Blanca

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 71660 | 45.81% |
| Frente De Todos | 67245 | 42.99% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8628 | 5.52% |
| Frente Nos | 6441 | 4.12% |
| Movimiento De Avanzada Socialista | 1664 | 1.06% |
| Frente Patriota | 786 | 0.50% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 51680 | 30.68% |
| Union Por La Patria | 48107 | 28.56% |
| La Libertad Avanza | 42384 | 25.16% |
| Resto | 18824 | 11.17% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4884 | 2.90% |
| Principios Y Valores | 1242 | 0.74% |
| Movimiento Libres Del Sur | 719 | 0.43% |
| Politica Obrera | 630 | 0.37% |

### Sección: Berisso

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 35824 | 65.84% |
| Juntos Por El Cambio | 13554 | 24.91% |
| Consenso Federal | 2760 | 5.07% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1765 | 3.24% |
| Movimiento De Avanzada Socialista | 504 | 0.93% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 18560 | 35.41% |
| Juntos Por El Cambio | 15541 | 29.65% |
| La Libertad Avanza | 9373 | 17.88% |
| Resto | 5326 | 10.16% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2357 | 4.50% |
| Principios Y Valores | 583 | 1.11% |
| Movimiento Libres Del Sur | 289 | 0.55% |
| Movimiento De Avanzada Socialista | 231 | 0.44% |
| Politica Obrera | 160 | 0.31% |

### Sección: Campana

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 27792 | 47.69% |
| Frente De Todos | 23040 | 39.53% |
| Consenso Federal | 3995 | 6.85% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1889 | 3.24% |
| Frente Nos | 1565 | 2.69% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 32660 | 53.57% |
| Union Por La Patria | 12905 | 21.17% |
| La Libertad Avanza | 7253 | 11.90% |
| Resto | 6375 | 10.46% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1127 | 1.85% |
| Principios Y Valores | 290 | 0.48% |
| Movimiento Libres Del Sur | 237 | 0.39% |
| Politica Obrera | 117 | 0.19% |

### Sección: Escobar

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 82919 | 67.74% |
| Juntos Por El Cambio | 28763 | 23.50% |
| Consenso Federal | 4615 | 3.77% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3037 | 2.48% |
| Frente Nos | 2374 | 1.94% |
| Movimiento De Avanzada Socialista | 695 | 0.57% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 54188 | 41.17% |
| La Libertad Avanza | 28369 | 21.55% |
| Juntos Por El Cambio | 27187 | 20.65% |
| Resto | 16514 | 12.55% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3324 | 2.53% |
| Principios Y Valores | 537 | 0.41% |
| Movimiento Libres Del Sur | 507 | 0.39% |
| Movimiento De Avanzada Socialista | 423 | 0.32% |
| Frente Patriota Federal | 336 | 0.26% |
| Politica Obrera | 250 | 0.19% |

### Sección: General Pueyrredón

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 132440 | 39.16% |
| Frente De Todos | 104135 | 30.79% |
| Accion Marplatense-Am | 65594 | 19.40% |
| Consenso Federal | 17662 | 5.22% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9363 | 2.77% |
| Frente Nos | 5315 | 1.57% |
| Movimiento De Avanzada Socialista | 2795 | 0.83% |
| Frente Patriota | 867 | 0.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 128787 | 36.49% |
| Alianza Encuentro Marplatense | 100447 | 28.46% |
| La Libertad Avanza | 70293 | 19.92% |
| Resto | 38121 | 10.80% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9305 | 2.64% |
| Movimiento Libres Del Sur | 2337 | 0.66% |
| Principios Y Valores | 1649 | 0.47% |
| Movimiento De Avanzada Socialista | 1290 | 0.37% |
| Politica Obrera | 730 | 0.21% |

### Sección: General Rodríguez

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 30380 | 56.82% |
| Juntos Por El Cambio | 19574 | 36.61% |
| Consenso Federal | 1859 | 3.48% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1168 | 2.18% |
| Movimiento De Avanzada Socialista | 489 | 0.91% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 23918 | 39.67% |
| Juntos Por El Cambio | 18476 | 30.64% |
| La Libertad Avanza | 9817 | 16.28% |
| Resto | 6084 | 10.09% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1270 | 2.11% |
| Principios Y Valores | 297 | 0.49% |
| Movimiento Libres Del Sur | 254 | 0.42% |
| Movimiento De Avanzada Socialista | 179 | 0.30% |

### Sección: Junín

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 26262 | 51.43% |
| Juntos Por El Cambio | 20010 | 39.19% |
| Consenso Federal | 3087 | 6.05% |
| Frente De Izquierda Y De Trabajadores - Unidad | 896 | 1.75% |
| Frente Nos | 603 | 1.18% |
| Movimiento De Avanzada Socialista | 207 | 0.41% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 22958 | 43.24% |
| Union Por La Patria | 14319 | 26.97% |
| La Libertad Avanza | 8424 | 15.87% |
| Resto | 5951 | 11.21% |
| Principios Y Valores | 722 | 1.36% |
| Frente De Izquierda Y De Trabajadores - Unidad | 553 | 1.04% |
| Movimiento Libres Del Sur | 171 | 0.32% |

### Sección: La Plata

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 183198 | 48.21% |
| Juntos Por El Cambio | 142998 | 37.63% |
| Consenso Federal | 25609 | 6.74% |
| Frente De Izquierda Y De Trabajadores - Unidad | 16356 | 4.30% |
| Frente Nos | 6709 | 1.77% |
| Movimiento De Avanzada Socialista | 4050 | 1.07% |
| Frente Patriota | 1094 | 0.29% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 145707 | 38.53% |
| Union Por La Patria | 112214 | 29.67% |
| La Libertad Avanza | 64012 | 16.93% |
| Resto | 33268 | 8.80% |
| Frente De Izquierda Y De Trabajadores - Unidad | 16119 | 4.26% |
| Principios Y Valores | 2099 | 0.56% |
| Movimiento De Avanzada Socialista | 1760 | 0.47% |
| Movimiento Libres Del Sur | 1725 | 0.46% |
| Politica Obrera | 657 | 0.17% |
| Frente Patriota Federal | 616 | 0.16% |

### Sección: Luján

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 30960 | 53.70% |
| Juntos Por El Cambio | 21514 | 37.32% |
| Consenso Federal | 3660 | 6.35% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1521 | 2.64% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 31155 | 51.66% |
| Juntos Por El Cambio | 13362 | 22.16% |
| La Libertad Avanza | 8441 | 14.00% |
| Resto | 5805 | 9.63% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1159 | 1.92% |
| Movimiento Libres Del Sur | 198 | 0.33% |
| Frente Patriota Federal | 183 | 0.30% |

### Sección: Necochea

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 31154 | 59.22% |
| Frente De Todos | 16161 | 30.72% |
| Consenso Federal | 3010 | 5.72% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1525 | 2.90% |
| Frente Nos | 759 | 1.44% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 11058 | 35.10% |
| Union Por La Patria | 7588 | 24.09% |
| La Libertad Avanza | 7033 | 22.32% |
| Resto | 4933 | 15.66% |
| Frente De Izquierda Y De Trabajadores - Unidad | 650 | 2.06% |
| Movimiento Libres Del Sur | 133 | 0.42% |
| Principios Y Valores | 110 | 0.35% |

### Sección: Olavarría

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 23858 | 39.08% |
| Frente De Todos | 21254 | 34.81% |
| Consenso Federal | 13031 | 21.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1730 | 2.83% |
| Frente Nos | 814 | 1.33% |
| Movimiento De Avanzada Socialista | 367 | 0.60% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 22901 | 34.69% |
| Juntos Por El Cambio | 20879 | 31.62% |
| La Libertad Avanza | 13783 | 20.88% |
| Resto | 6448 | 9.77% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1341 | 2.03% |
| Principios Y Valores | 506 | 0.77% |
| Movimiento De Avanzada Socialista | 166 | 0.25% |

### Sección: Pergamino

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 25917 | 46.31% |
| Frente De Todos | 21012 | 37.55% |
| Consenso Federal | 7766 | 13.88% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1267 | 2.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 27365 | 48.53% |
| Union Por La Patria | 11017 | 19.54% |
| La Libertad Avanza | 10248 | 18.17% |
| Resto | 6160 | 10.92% |
| Movimiento Libres Del Sur | 792 | 1.40% |
| Frente De Izquierda Y De Trabajadores - Unidad | 346 | 0.61% |
| Principios Y Valores | 250 | 0.44% |
| Politica Obrera | 215 | 0.38% |

### Sección: Pilar

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 61853 | 51.42% |
| Juntos Por El Cambio | 45447 | 37.78% |
| Consenso Federal | 7251 | 6.03% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3469 | 2.88% |
| Frente Nos | 1592 | 1.32% |
| Movimiento De Avanzada Socialista | 669 | 0.56% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 93957 | 52.92% |
| Juntos Por El Cambio | 40346 | 22.73% |
| Resto | 19218 | 10.82% |
| La Libertad Avanza | 17439 | 9.82% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3857 | 2.17% |
| Principios Y Valores | 835 | 0.47% |
| Movimiento Libres Del Sur | 609 | 0.34% |
| Frente Patriota Federal | 535 | 0.30% |
| Movimiento De Avanzada Socialista | 407 | 0.23% |
| Politica Obrera | 337 | 0.19% |

### Sección: San Nicolás

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 40477 | 49.55% |
| Frente De Todos | 30308 | 37.10% |
| Consenso Federal | 5535 | 6.78% |
| Frente Nos | 2796 | 3.42% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2069 | 2.53% |
| Movimiento De Avanzada Socialista | 506 | 0.62% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 33576 | 40.59% |
| Union Por La Patria | 20484 | 24.76% |
| La Libertad Avanza | 14300 | 17.29% |
| Resto | 11203 | 13.54% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1930 | 2.33% |
| Principios Y Valores | 738 | 0.89% |
| Movimiento Libres Del Sur | 340 | 0.41% |
| Frente Patriota Federal | 150 | 0.18% |

### Sección: Tandil

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 41144 | 56.01% |
| Frente De Todos | 26532 | 36.12% |
| Consenso Federal | 3703 | 5.04% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1635 | 2.23% |
| Movimiento De Avanzada Socialista | 438 | 0.60% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 34392 | 45.39% |
| Union Por La Patria | 18516 | 24.44% |
| La Libertad Avanza | 13054 | 17.23% |
| Resto | 7983 | 10.54% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1185 | 1.56% |
| Principios Y Valores | 455 | 0.60% |
| Movimiento Libres Del Sur | 189 | 0.25% |

### Sección: Zárate

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 38046 | 57.98% |
| Juntos Por El Cambio | 20721 | 31.58% |
| Consenso Federal | 3884 | 5.92% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1887 | 2.88% |
| Frente Nos | 1086 | 1.65% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 27249 | 40.33% |
| Juntos Por El Cambio | 19533 | 28.91% |
| La Libertad Avanza | 11687 | 17.30% |
| Resto | 7047 | 10.43% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1114 | 1.65% |
| Principios Y Valores | 419 | 0.62% |
| Movimiento Libres Del Sur | 413 | 0.61% |
| Frente Patriota Federal | 108 | 0.16% |


In [90]:
# Grouping by the mentioned columns and getting the index of the maximum votes within each group
idx = tabla.groupby(['seccion_id', 'seccion_nombre', 'Region', 'eleccion_tag'])['votos_cantidad'].idxmax()

# Using the index to extract the corresponding rows from the original DataFrame
most_votes = tabla.loc[idx].reset_index(drop=True)

# Now, 'most_votes' contains the agrupaciones with the most votes for every place in the two elections


In [91]:
table_u = most_votes.set_index(['seccion_id', 'eleccion_tag'])['agrupacion_nombre_'].unstack()

# Obtenemos las combinaciones únicas de resultados del 2019 y 2023
combinaciones = table_u.groupby(['PASO19n', 'PASO23n']).size().reset_index()

# Función para mostrar la tabla de votos y porcentajes por elección
def mostrar_tabla(data, eleccion):
    df = data[data['eleccion_tag'] == eleccion]
    df = df.sort_values(by='votos_cantidad', ascending=False)
    table_md = "| Agrupación | Votos | Porcentaje |\n|---|---|---|\n"
    for _, row in df.iterrows():
        table_md += f"| {row['agrupacion_nombre_']} | {row['votos_cantidad']} | {row['votos_porcentaje']:.2f}% |\n"
    return table_md

output_str = ""

# Iteramos sobre las combinaciones
for _, comb in combinaciones.iterrows():
    ganador_19 = comb['PASO19n']
    ganador_23 = comb['PASO23n']
    
    # Filtramos los datos por combinación
    data_combinacion = table_u[(table_u['PASO19n'] == ganador_19) & (table_u['PASO23n'] == ganador_23)]
    
    output_str += f"\n## Combinación: {ganador_19} (2019) - {ganador_23} (2023)\n"
    
    # Para cada sección en la combinación, mostramos la tabla de votos y porcentajes
    for seccion, data_seccion in data_combinacion.groupby('seccion_id'):
        row = data_seccion.iloc[0]
        output_str += f"\n### Partido: {tabla[tabla['seccion_id'] == seccion]['seccion_nombre'].unique()[0]}\n"
        
        # Tabla 2019
        output_str += "\n**Elección 2019**\n"
        output_str += mostrar_tabla(tabla[tabla['seccion_id'] == seccion], 'PASO19n')
        
        # Tabla 2023
        output_str += "\n**Elección 2023**\n"
        output_str += mostrar_tabla(tabla[tabla['seccion_id'] == seccion], 'PASO23n')

# Mostramos las tablas
display(Markdown(output_str))



## Combinación: Frente De Todos (2019) - Juntos Por El Cambio (2023)

### Partido: Junín

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 26262 | 51.43% |
| Juntos Por El Cambio | 20010 | 39.19% |
| Consenso Federal | 3087 | 6.05% |
| Frente De Izquierda Y De Trabajadores - Unidad | 896 | 1.75% |
| Frente Nos | 603 | 1.18% |
| Movimiento De Avanzada Socialista | 207 | 0.41% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 22958 | 43.24% |
| Union Por La Patria | 14319 | 26.97% |
| La Libertad Avanza | 8424 | 15.87% |
| Resto | 5951 | 11.21% |
| Principios Y Valores | 722 | 1.36% |
| Frente De Izquierda Y De Trabajadores - Unidad | 553 | 1.04% |
| Movimiento Libres Del Sur | 171 | 0.32% |

### Partido: La Plata

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 183198 | 48.21% |
| Juntos Por El Cambio | 142998 | 37.63% |
| Consenso Federal | 25609 | 6.74% |
| Frente De Izquierda Y De Trabajadores - Unidad | 16356 | 4.30% |
| Frente Nos | 6709 | 1.77% |
| Movimiento De Avanzada Socialista | 4050 | 1.07% |
| Frente Patriota | 1094 | 0.29% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 145707 | 38.53% |
| Union Por La Patria | 112214 | 29.67% |
| La Libertad Avanza | 64012 | 16.93% |
| Resto | 33268 | 8.80% |
| Frente De Izquierda Y De Trabajadores - Unidad | 16119 | 4.26% |
| Principios Y Valores | 2099 | 0.56% |
| Movimiento De Avanzada Socialista | 1760 | 0.47% |
| Movimiento Libres Del Sur | 1725 | 0.46% |
| Politica Obrera | 657 | 0.17% |
| Frente Patriota Federal | 616 | 0.16% |

### Partido: Morón

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 91484 | 46.74% |
| Juntos Por El Cambio | 75401 | 38.53% |
| Consenso Federal | 15097 | 7.71% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8326 | 4.25% |
| Frente Nos | 3488 | 1.78% |
| Movimiento De Avanzada Socialista | 1921 | 0.98% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 65435 | 33.63% |
| Union Por La Patria | 61399 | 31.55% |
| La Libertad Avanza | 39405 | 20.25% |
| Resto | 16370 | 8.41% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7687 | 3.95% |
| Principios Y Valores | 1976 | 1.02% |
| Movimiento De Avanzada Socialista | 816 | 0.42% |
| Movimiento Libres Del Sur | 688 | 0.35% |
| Politica Obrera | 421 | 0.22% |
| Frente Patriota Federal | 403 | 0.21% |

### Partido: Tres De Febrero

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 97576 | 49.43% |
| Juntos Por El Cambio | 72756 | 36.86% |
| Consenso Federal | 14437 | 7.31% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8254 | 4.18% |
| Frente Nos | 2614 | 1.32% |
| Movimiento De Avanzada Socialista | 1756 | 0.89% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 80356 | 40.85% |
| Union Por La Patria | 54591 | 27.75% |
| La Libertad Avanza | 33161 | 16.86% |
| Resto | 18539 | 9.43% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6574 | 3.34% |
| Principios Y Valores | 1638 | 0.83% |
| Movimiento Libres Del Sur | 749 | 0.38% |
| Movimiento De Avanzada Socialista | 670 | 0.34% |
| Politica Obrera | 419 | 0.21% |

### Partido: Ituzaingo

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 46493 | 48.18% |
| Juntos Por El Cambio | 33763 | 34.99% |
| Consenso Federal | 9080 | 9.41% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4529 | 4.69% |
| Frente Nos | 1604 | 1.66% |
| Movimiento De Avanzada Socialista | 1020 | 1.06% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 33836 | 32.32% |
| Union Por La Patria | 33360 | 31.87% |
| La Libertad Avanza | 20384 | 19.47% |
| Resto | 10048 | 9.60% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4537 | 4.33% |
| Principios Y Valores | 907 | 0.87% |
| Movimiento De Avanzada Socialista | 543 | 0.52% |
| Movimiento Libres Del Sur | 401 | 0.38% |
| Frente Patriota Federal | 374 | 0.36% |
| Politica Obrera | 295 | 0.28% |

## Combinación: Frente De Todos (2019) - Union Por La Patria (2023)

### Partido: Almirante Brown

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 188633 | 62.28% |
| Juntos Por El Cambio | 72967 | 24.09% |
| Consenso Federal | 19969 | 6.59% |
| Frente De Izquierda Y De Trabajadores - Unidad | 13586 | 4.49% |
| Frente Nos | 3626 | 1.20% |
| Movimiento De Avanzada Socialista | 3148 | 1.04% |
| Frente Patriota | 931 | 0.31% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 118118 | 40.37% |
| La Libertad Avanza | 66617 | 22.77% |
| Juntos Por El Cambio | 52926 | 18.09% |
| Resto | 35435 | 12.11% |
| Frente De Izquierda Y De Trabajadores - Unidad | 12364 | 4.23% |
| Principios Y Valores | 2651 | 0.91% |
| Movimiento Libres Del Sur | 1484 | 0.51% |
| Movimiento De Avanzada Socialista | 1406 | 0.48% |
| Frente Patriota Federal | 856 | 0.29% |
| Politica Obrera | 750 | 0.26% |

### Partido: Avellaneda

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 122160 | 59.85% |
| Juntos Por El Cambio | 62033 | 30.39% |
| Consenso Federal | 11401 | 5.59% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6475 | 3.17% |
| Movimiento De Avanzada Socialista | 1517 | 0.74% |
| Frente Patriota | 540 | 0.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 89113 | 43.50% |
| Juntos Por El Cambio | 51812 | 25.29% |
| La Libertad Avanza | 33143 | 16.18% |
| Resto | 20839 | 10.17% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6531 | 3.19% |
| Principios Y Valores | 1495 | 0.73% |
| Movimiento De Avanzada Socialista | 694 | 0.34% |
| Movimiento Libres Del Sur | 481 | 0.23% |
| Frente Patriota Federal | 391 | 0.19% |
| Politica Obrera | 370 | 0.18% |

### Partido: Berazategui

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 123847 | 67.95% |
| Juntos Por El Cambio | 38875 | 21.33% |
| Consenso Federal | 8342 | 4.58% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5925 | 3.25% |
| Frente Nos | 3489 | 1.91% |
| Movimiento De Avanzada Socialista | 1286 | 0.71% |
| Frente Patriota | 494 | 0.27% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 81885 | 42.86% |
| La Libertad Avanza | 38825 | 20.32% |
| Juntos Por El Cambio | 37514 | 19.64% |
| Resto | 22559 | 11.81% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6513 | 3.41% |
| Principios Y Valores | 1870 | 0.98% |
| Movimiento De Avanzada Socialista | 717 | 0.38% |
| Movimiento Libres Del Sur | 667 | 0.35% |
| Politica Obrera | 496 | 0.26% |

### Partido: Berisso

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 35824 | 65.84% |
| Juntos Por El Cambio | 13554 | 24.91% |
| Consenso Federal | 2760 | 5.07% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1765 | 3.24% |
| Movimiento De Avanzada Socialista | 504 | 0.93% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 18560 | 35.41% |
| Juntos Por El Cambio | 15541 | 29.65% |
| La Libertad Avanza | 9373 | 17.88% |
| Resto | 5326 | 10.16% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2357 | 4.50% |
| Principios Y Valores | 583 | 1.11% |
| Movimiento Libres Del Sur | 289 | 0.55% |
| Movimiento De Avanzada Socialista | 231 | 0.44% |
| Politica Obrera | 160 | 0.31% |

### Partido: Escobar

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 82919 | 67.74% |
| Juntos Por El Cambio | 28763 | 23.50% |
| Consenso Federal | 4615 | 3.77% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3037 | 2.48% |
| Frente Nos | 2374 | 1.94% |
| Movimiento De Avanzada Socialista | 695 | 0.57% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 54188 | 41.17% |
| La Libertad Avanza | 28369 | 21.55% |
| Juntos Por El Cambio | 27187 | 20.65% |
| Resto | 16514 | 12.55% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3324 | 2.53% |
| Principios Y Valores | 537 | 0.41% |
| Movimiento Libres Del Sur | 507 | 0.39% |
| Movimiento De Avanzada Socialista | 423 | 0.32% |
| Frente Patriota Federal | 336 | 0.26% |
| Politica Obrera | 250 | 0.19% |

### Partido: Esteban Echeverría

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 96614 | 59.15% |
| Juntos Por El Cambio | 44889 | 27.48% |
| Consenso Federal | 11380 | 6.97% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5985 | 3.66% |
| Frente Nos | 2622 | 1.61% |
| Movimiento De Avanzada Socialista | 1265 | 0.77% |
| Frente Patriota | 571 | 0.35% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 59071 | 34.81% |
| La Libertad Avanza | 40140 | 23.65% |
| Juntos Por El Cambio | 39937 | 23.53% |
| Resto | 20361 | 12.00% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6258 | 3.69% |
| Principios Y Valores | 1299 | 0.77% |
| Movimiento Libres Del Sur | 1061 | 0.63% |
| Movimiento De Avanzada Socialista | 652 | 0.38% |
| Frente Patriota Federal | 520 | 0.31% |
| Politica Obrera | 420 | 0.25% |

### Partido: Florencio Varela

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 103294 | 52.19% |
| Juntos Por El Cambio | 55011 | 27.79% |
| Consenso Federal | 26496 | 13.39% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7149 | 3.61% |
| Frente Nos | 3478 | 1.76% |
| Movimiento De Avanzada Socialista | 1669 | 0.84% |
| Frente Patriota | 831 | 0.42% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 82562 | 38.21% |
| La Libertad Avanza | 48447 | 22.42% |
| Juntos Por El Cambio | 38615 | 17.87% |
| Resto | 32548 | 15.06% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8454 | 3.91% |
| Principios Y Valores | 2128 | 0.98% |
| Movimiento Libres Del Sur | 1143 | 0.53% |
| Movimiento De Avanzada Socialista | 942 | 0.44% |
| Frente Patriota Federal | 652 | 0.30% |
| Politica Obrera | 567 | 0.26% |

### Partido: General Rodríguez

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 30380 | 56.82% |
| Juntos Por El Cambio | 19574 | 36.61% |
| Consenso Federal | 1859 | 3.48% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1168 | 2.18% |
| Movimiento De Avanzada Socialista | 489 | 0.91% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 23918 | 39.67% |
| Juntos Por El Cambio | 18476 | 30.64% |
| La Libertad Avanza | 9817 | 16.28% |
| Resto | 6084 | 10.09% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1270 | 2.11% |
| Principios Y Valores | 297 | 0.49% |
| Movimiento Libres Del Sur | 254 | 0.42% |
| Movimiento De Avanzada Socialista | 179 | 0.30% |

### Partido: General San Martín

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 125320 | 55.35% |
| Juntos Por El Cambio | 70243 | 31.03% |
| Consenso Federal | 14972 | 6.61% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9500 | 4.20% |
| Frente Nos | 3672 | 1.62% |
| Movimiento De Avanzada Socialista | 1938 | 0.86% |
| Frente Patriota | 761 | 0.34% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 88680 | 37.59% |
| Juntos Por El Cambio | 66607 | 28.23% |
| La Libertad Avanza | 47941 | 20.32% |
| Resto | 21108 | 8.95% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7209 | 3.06% |
| Principios Y Valores | 1834 | 0.78% |
| Movimiento Libres Del Sur | 1117 | 0.47% |
| Movimiento De Avanzada Socialista | 889 | 0.38% |
| Politica Obrera | 538 | 0.23% |

### Partido: La Matanza

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 472575 | 64.65% |
| Juntos Por El Cambio | 160529 | 21.96% |
| Consenso Federal | 45946 | 6.29% |
| Frente De Izquierda Y De Trabajadores - Unidad | 30538 | 4.18% |
| Frente Nos | 13982 | 1.91% |
| Movimiento De Avanzada Socialista | 5195 | 0.71% |
| Frente Patriota | 2155 | 0.29% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 280043 | 39.77% |
| La Libertad Avanza | 148003 | 21.02% |
| Juntos Por El Cambio | 146442 | 20.80% |
| Resto | 82047 | 11.65% |
| Frente De Izquierda Y De Trabajadores - Unidad | 32213 | 4.57% |
| Principios Y Valores | 6091 | 0.86% |
| Movimiento Libres Del Sur | 3405 | 0.48% |
| Movimiento De Avanzada Socialista | 2334 | 0.33% |
| Frente Patriota Federal | 1808 | 0.26% |
| Politica Obrera | 1802 | 0.26% |

### Partido: Lanús

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 137462 | 51.12% |
| Juntos Por El Cambio | 98945 | 36.80% |
| Consenso Federal | 17052 | 6.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 10118 | 3.76% |
| Frente Nos | 3089 | 1.15% |
| Movimiento De Avanzada Socialista | 2218 | 0.82% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 95965 | 36.81% |
| Juntos Por El Cambio | 83133 | 31.89% |
| La Libertad Avanza | 45298 | 17.38% |
| Resto | 22063 | 8.46% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9439 | 3.62% |
| Principios Y Valores | 1949 | 0.75% |
| Movimiento Libres Del Sur | 1100 | 0.42% |
| Movimiento De Avanzada Socialista | 1040 | 0.40% |
| Politica Obrera | 683 | 0.26% |

### Partido: Lomas De Zamora

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 210394 | 61.04% |
| Juntos Por El Cambio | 91220 | 26.46% |
| Consenso Federal | 21552 | 6.25% |
| Frente De Izquierda Y De Trabajadores - Unidad | 13839 | 4.01% |
| Frente Nos | 4422 | 1.28% |
| Movimiento De Avanzada Socialista | 3261 | 0.95% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 138101 | 38.01% |
| Juntos Por El Cambio | 93174 | 25.65% |
| La Libertad Avanza | 73391 | 20.20% |
| Resto | 37637 | 10.36% |
| Frente De Izquierda Y De Trabajadores - Unidad | 13398 | 3.69% |
| Principios Y Valores | 2881 | 0.79% |
| Movimiento Libres Del Sur | 1620 | 0.45% |
| Movimiento De Avanzada Socialista | 1522 | 0.42% |
| Frente Patriota Federal | 795 | 0.22% |
| Politica Obrera | 770 | 0.21% |

### Partido: Luján

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 30960 | 53.70% |
| Juntos Por El Cambio | 21514 | 37.32% |
| Consenso Federal | 3660 | 6.35% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1521 | 2.64% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 31155 | 51.66% |
| Juntos Por El Cambio | 13362 | 22.16% |
| La Libertad Avanza | 8441 | 14.00% |
| Resto | 5805 | 9.63% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1159 | 1.92% |
| Movimiento Libres Del Sur | 198 | 0.33% |
| Frente Patriota Federal | 183 | 0.30% |

### Partido: Merlo

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 176748 | 77.70% |
| Juntos Por El Cambio | 23955 | 10.53% |
| Frente De Izquierda Y De Trabajadores - Unidad | 11131 | 4.89% |
| Consenso Federal | 10392 | 4.57% |
| Frente Nos | 2602 | 1.14% |
| Movimiento De Avanzada Socialista | 2048 | 0.90% |
| Frente Patriota | 591 | 0.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 73999 | 31.10% |
| La Libertad Avanza | 55527 | 23.34% |
| Juntos Por El Cambio | 52455 | 22.05% |
| Resto | 34653 | 14.56% |
| Frente De Izquierda Y De Trabajadores - Unidad | 12847 | 5.40% |
| Movimiento Libres Del Sur | 2890 | 1.21% |
| Principios Y Valores | 2251 | 0.95% |
| Movimiento De Avanzada Socialista | 1351 | 0.57% |
| Politica Obrera | 980 | 0.41% |
| Frente Patriota Federal | 969 | 0.41% |

### Partido: Moreno

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 155607 | 68.63% |
| Juntos Por El Cambio | 49095 | 21.65% |
| Consenso Federal | 10067 | 4.44% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7180 | 3.17% |
| Frente Nos | 2606 | 1.15% |
| Movimiento De Avanzada Socialista | 1513 | 0.67% |
| Frente Patriota | 657 | 0.29% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 113463 | 48.08% |
| La Libertad Avanza | 49823 | 21.11% |
| Juntos Por El Cambio | 35437 | 15.02% |
| Resto | 24633 | 10.44% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8115 | 3.44% |
| Movimiento Libres Del Sur | 1378 | 0.58% |
| Principios Y Valores | 1122 | 0.48% |
| Movimiento De Avanzada Socialista | 804 | 0.34% |
| Frente Patriota Federal | 621 | 0.26% |
| Politica Obrera | 582 | 0.25% |

### Partido: Pilar

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 61853 | 51.42% |
| Juntos Por El Cambio | 45447 | 37.78% |
| Consenso Federal | 7251 | 6.03% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3469 | 2.88% |
| Frente Nos | 1592 | 1.32% |
| Movimiento De Avanzada Socialista | 669 | 0.56% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 93957 | 52.92% |
| Juntos Por El Cambio | 40346 | 22.73% |
| Resto | 19218 | 10.82% |
| La Libertad Avanza | 17439 | 9.82% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3857 | 2.17% |
| Principios Y Valores | 835 | 0.47% |
| Movimiento Libres Del Sur | 609 | 0.34% |
| Frente Patriota Federal | 535 | 0.30% |
| Movimiento De Avanzada Socialista | 407 | 0.23% |
| Politica Obrera | 337 | 0.19% |

### Partido: Quilmes

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 175769 | 55.24% |
| Juntos Por El Cambio | 104654 | 32.89% |
| Consenso Federal | 21419 | 6.73% |
| Frente De Izquierda Y De Trabajadores - Unidad | 10240 | 3.22% |
| Frente Nos | 4090 | 1.29% |
| Movimiento De Avanzada Socialista | 1994 | 0.63% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 121126 | 37.00% |
| Juntos Por El Cambio | 103699 | 31.68% |
| La Libertad Avanza | 52858 | 16.15% |
| Resto | 34425 | 10.52% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9910 | 3.03% |
| Principios Y Valores | 1828 | 0.56% |
| Movimiento Libres Del Sur | 1210 | 0.37% |
| Movimiento De Avanzada Socialista | 1044 | 0.32% |
| Politica Obrera | 638 | 0.19% |
| Frente Patriota Federal | 631 | 0.19% |

### Partido: San Fernando

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 57377 | 64.50% |
| Juntos Por El Cambio | 23536 | 26.46% |
| Consenso Federal | 4761 | 5.35% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2654 | 2.98% |
| Movimiento De Avanzada Socialista | 634 | 0.71% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 46063 | 48.58% |
| Juntos Por El Cambio | 20951 | 22.10% |
| La Libertad Avanza | 14618 | 15.42% |
| Resto | 9763 | 10.30% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1910 | 2.01% |
| Movimiento Libres Del Sur | 657 | 0.69% |
| Principios Y Valores | 444 | 0.47% |
| Movimiento De Avanzada Socialista | 249 | 0.26% |
| Politica Obrera | 163 | 0.17% |

### Partido: Tigre

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 121630 | 56.39% |
| Juntos Por El Cambio | 67829 | 31.45% |
| Consenso Federal | 11534 | 5.35% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8413 | 3.90% |
| Frente Nos | 4731 | 2.19% |
| Movimiento De Avanzada Socialista | 1545 | 0.72% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 103033 | 45.28% |
| Juntos Por El Cambio | 52360 | 23.01% |
| La Libertad Avanza | 40177 | 17.66% |
| Resto | 22932 | 10.08% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6132 | 2.69% |
| Movimiento Libres Del Sur | 1061 | 0.47% |
| Principios Y Valores | 830 | 0.36% |
| Movimiento De Avanzada Socialista | 637 | 0.28% |
| Politica Obrera | 399 | 0.18% |

### Partido: Zárate

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 38046 | 57.98% |
| Juntos Por El Cambio | 20721 | 31.58% |
| Consenso Federal | 3884 | 5.92% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1887 | 2.88% |
| Frente Nos | 1086 | 1.65% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 27249 | 40.33% |
| Juntos Por El Cambio | 19533 | 28.91% |
| La Libertad Avanza | 11687 | 17.30% |
| Resto | 7047 | 10.43% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1114 | 1.65% |
| Principios Y Valores | 419 | 0.62% |
| Movimiento Libres Del Sur | 413 | 0.61% |
| Frente Patriota Federal | 108 | 0.16% |

### Partido: J. C. Paz

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 88667 | 61.88% |
| Juntos Por El Cambio | 34687 | 24.21% |
| Consenso Federal | 9081 | 6.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6309 | 4.40% |
| Frente Nos | 3242 | 2.26% |
| Movimiento De Avanzada Socialista | 1305 | 0.91% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 51254 | 35.75% |
| La Libertad Avanza | 32913 | 22.95% |
| Juntos Por El Cambio | 29381 | 20.49% |
| Resto | 20609 | 14.37% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6025 | 4.20% |
| Principios Y Valores | 1049 | 0.73% |
| Movimiento Libres Del Sur | 679 | 0.47% |
| Movimiento De Avanzada Socialista | 642 | 0.45% |
| Frente Patriota Federal | 435 | 0.30% |
| Politica Obrera | 396 | 0.28% |

### Partido: Malvinas Argentinas

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 122679 | 64.31% |
| Juntos Por El Cambio | 50029 | 26.22% |
| Consenso Federal | 8201 | 4.30% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5460 | 2.86% |
| Frente Nos | 3292 | 1.73% |
| Movimiento De Avanzada Socialista | 1115 | 0.58% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 85884 | 45.03% |
| La Libertad Avanza | 38917 | 20.41% |
| Juntos Por El Cambio | 33974 | 17.81% |
| Resto | 21324 | 11.18% |
| Frente De Izquierda Y De Trabajadores - Unidad | 6169 | 3.23% |
| Movimiento Libres Del Sur | 1483 | 0.78% |
| Principios Y Valores | 1274 | 0.67% |
| Movimiento De Avanzada Socialista | 738 | 0.39% |
| Frente Patriota Federal | 505 | 0.26% |
| Politica Obrera | 446 | 0.23% |

### Partido: Ezeiza

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 57314 | 68.47% |
| Juntos Por El Cambio | 19084 | 22.80% |
| Frente De Izquierda Y De Trabajadores - Unidad | 3245 | 3.88% |
| Consenso Federal | 2563 | 3.06% |
| Frente Nos | 968 | 1.16% |
| Movimiento De Avanzada Socialista | 528 | 0.63% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 45556 | 46.44% |
| La Libertad Avanza | 20006 | 20.39% |
| Juntos Por El Cambio | 15705 | 16.01% |
| Resto | 12405 | 12.65% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2643 | 2.69% |
| Principios Y Valores | 528 | 0.54% |
| Frente Patriota Federal | 374 | 0.38% |
| Movimiento Libres Del Sur | 364 | 0.37% |
| Movimiento De Avanzada Socialista | 274 | 0.28% |
| Politica Obrera | 245 | 0.25% |

### Partido: Hurlingham

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Frente De Todos | 59719 | 55.18% |
| Juntos Por El Cambio | 32357 | 29.90% |
| Consenso Federal | 7853 | 7.26% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4560 | 4.21% |
| Frente Nos | 2824 | 2.61% |
| Movimiento De Avanzada Socialista | 908 | 0.84% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 53577 | 48.75% |
| Juntos Por El Cambio | 25598 | 23.29% |
| La Libertad Avanza | 17148 | 15.60% |
| Resto | 8972 | 8.16% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2852 | 2.60% |
| Principios Y Valores | 611 | 0.56% |
| Movimiento Libres Del Sur | 468 | 0.43% |
| Movimiento De Avanzada Socialista | 336 | 0.31% |
| Politica Obrera | 333 | 0.30% |

## Combinación: Juntos Por El Cambio (2019) - Juntos Por El Cambio (2023)

### Partido: Bahía Blanca

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 71660 | 45.81% |
| Frente De Todos | 67245 | 42.99% |
| Frente De Izquierda Y De Trabajadores - Unidad | 8628 | 5.52% |
| Frente Nos | 6441 | 4.12% |
| Movimiento De Avanzada Socialista | 1664 | 1.06% |
| Frente Patriota | 786 | 0.50% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 51680 | 30.68% |
| Union Por La Patria | 48107 | 28.56% |
| La Libertad Avanza | 42384 | 25.16% |
| Resto | 18824 | 11.17% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4884 | 2.90% |
| Principios Y Valores | 1242 | 0.74% |
| Movimiento Libres Del Sur | 719 | 0.43% |
| Politica Obrera | 630 | 0.37% |

### Partido: Campana

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 27792 | 47.69% |
| Frente De Todos | 23040 | 39.53% |
| Consenso Federal | 3995 | 6.85% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1889 | 3.24% |
| Frente Nos | 1565 | 2.69% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 32660 | 53.57% |
| Union Por La Patria | 12905 | 21.17% |
| La Libertad Avanza | 7253 | 11.90% |
| Resto | 6375 | 10.46% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1127 | 1.85% |
| Principios Y Valores | 290 | 0.48% |
| Movimiento Libres Del Sur | 237 | 0.39% |
| Politica Obrera | 117 | 0.19% |

### Partido: General Pueyrredón

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 132440 | 39.16% |
| Frente De Todos | 104135 | 30.79% |
| Accion Marplatense-Am | 65594 | 19.40% |
| Consenso Federal | 17662 | 5.22% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9363 | 2.77% |
| Frente Nos | 5315 | 1.57% |
| Movimiento De Avanzada Socialista | 2795 | 0.83% |
| Frente Patriota | 867 | 0.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 128787 | 36.49% |
| Alianza Encuentro Marplatense | 100447 | 28.46% |
| La Libertad Avanza | 70293 | 19.92% |
| Resto | 38121 | 10.80% |
| Frente De Izquierda Y De Trabajadores - Unidad | 9305 | 2.64% |
| Movimiento Libres Del Sur | 2337 | 0.66% |
| Principios Y Valores | 1649 | 0.47% |
| Movimiento De Avanzada Socialista | 1290 | 0.37% |
| Politica Obrera | 730 | 0.21% |

### Partido: San Miguel

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 80257 | 51.07% |
| Frente De Todos | 63979 | 40.71% |
| Consenso Federal | 6817 | 4.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4912 | 3.13% |
| Movimiento De Avanzada Socialista | 1186 | 0.75% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 76298 | 44.88% |
| Union Por La Patria | 44659 | 26.27% |
| La Libertad Avanza | 25818 | 15.19% |
| Resto | 16466 | 9.69% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4681 | 2.75% |
| Principios Y Valores | 684 | 0.40% |
| Movimiento De Avanzada Socialista | 507 | 0.30% |
| Movimiento Libres Del Sur | 356 | 0.21% |
| Politica Obrera | 300 | 0.18% |
| Frente Patriota Federal | 244 | 0.14% |

### Partido: Necochea

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 31154 | 59.22% |
| Frente De Todos | 16161 | 30.72% |
| Consenso Federal | 3010 | 5.72% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1525 | 2.90% |
| Frente Nos | 759 | 1.44% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 11058 | 35.10% |
| Union Por La Patria | 7588 | 24.09% |
| La Libertad Avanza | 7033 | 22.32% |
| Resto | 4933 | 15.66% |
| Frente De Izquierda Y De Trabajadores - Unidad | 650 | 2.06% |
| Movimiento Libres Del Sur | 133 | 0.42% |
| Principios Y Valores | 110 | 0.35% |

### Partido: Pergamino

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 25917 | 46.31% |
| Frente De Todos | 21012 | 37.55% |
| Consenso Federal | 7766 | 13.88% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1267 | 2.26% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 27365 | 48.53% |
| Union Por La Patria | 11017 | 19.54% |
| La Libertad Avanza | 10248 | 18.17% |
| Resto | 6160 | 10.92% |
| Movimiento Libres Del Sur | 792 | 1.40% |
| Frente De Izquierda Y De Trabajadores - Unidad | 346 | 0.61% |
| Principios Y Valores | 250 | 0.44% |
| Politica Obrera | 215 | 0.38% |

### Partido: San Isidro

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 83601 | 55.24% |
| Frente De Todos | 52864 | 34.93% |
| Consenso Federal | 7946 | 5.25% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5642 | 3.73% |
| Movimiento De Avanzada Socialista | 1281 | 0.85% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 106459 | 54.98% |
| Union Por La Patria | 32291 | 16.68% |
| La Libertad Avanza | 30966 | 15.99% |
| Resto | 15766 | 8.14% |
| Frente De Izquierda Y De Trabajadores - Unidad | 4933 | 2.55% |
| Principios Y Valores | 1301 | 0.67% |
| Movimiento Libres Del Sur | 911 | 0.47% |
| Movimiento De Avanzada Socialista | 592 | 0.31% |
| Politica Obrera | 396 | 0.20% |

### Partido: San Nicolás

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 40477 | 49.55% |
| Frente De Todos | 30308 | 37.10% |
| Consenso Federal | 5535 | 6.78% |
| Frente Nos | 2796 | 3.42% |
| Frente De Izquierda Y De Trabajadores - Unidad | 2069 | 2.53% |
| Movimiento De Avanzada Socialista | 506 | 0.62% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 33576 | 40.59% |
| Union Por La Patria | 20484 | 24.76% |
| La Libertad Avanza | 14300 | 17.29% |
| Resto | 11203 | 13.54% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1930 | 2.33% |
| Principios Y Valores | 738 | 0.89% |
| Movimiento Libres Del Sur | 340 | 0.41% |
| Frente Patriota Federal | 150 | 0.18% |

### Partido: Tandil

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 41144 | 56.01% |
| Frente De Todos | 26532 | 36.12% |
| Consenso Federal | 3703 | 5.04% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1635 | 2.23% |
| Movimiento De Avanzada Socialista | 438 | 0.60% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 34392 | 45.39% |
| Union Por La Patria | 18516 | 24.44% |
| La Libertad Avanza | 13054 | 17.23% |
| Resto | 7983 | 10.54% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1185 | 1.56% |
| Principios Y Valores | 455 | 0.60% |
| Movimiento Libres Del Sur | 189 | 0.25% |

### Partido: Vicente López

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 94223 | 57.14% |
| Frente De Todos | 44810 | 27.17% |
| Consenso Federal | 13707 | 8.31% |
| Frente De Izquierda Y De Trabajadores - Unidad | 7467 | 4.53% |
| Frente Nos | 2876 | 1.74% |
| Movimiento De Avanzada Socialista | 1813 | 1.10% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 74038 | 47.62% |
| Union Por La Patria | 32107 | 20.65% |
| La Libertad Avanza | 27419 | 17.63% |
| Resto | 13085 | 8.42% |
| Frente De Izquierda Y De Trabajadores - Unidad | 5994 | 3.85% |
| Principios Y Valores | 1214 | 0.78% |
| Movimiento De Avanzada Socialista | 653 | 0.42% |
| Movimiento Libres Del Sur | 436 | 0.28% |
| Politica Obrera | 289 | 0.19% |
| Frente Patriota Federal | 255 | 0.16% |

## Combinación: Juntos Por El Cambio (2019) - Union Por La Patria (2023)

### Partido: Olavarría

**Elección 2019**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Juntos Por El Cambio | 23858 | 39.08% |
| Frente De Todos | 21254 | 34.81% |
| Consenso Federal | 13031 | 21.34% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1730 | 2.83% |
| Frente Nos | 814 | 1.33% |
| Movimiento De Avanzada Socialista | 367 | 0.60% |

**Elección 2023**
| Agrupación | Votos | Porcentaje |
|---|---|---|
| Union Por La Patria | 22901 | 34.69% |
| Juntos Por El Cambio | 20879 | 31.62% |
| La Libertad Avanza | 13783 | 20.88% |
| Resto | 6448 | 9.77% |
| Frente De Izquierda Y De Trabajadores - Unidad | 1341 | 2.03% |
| Principios Y Valores | 506 | 0.77% |
| Movimiento De Avanzada Socialista | 166 | 0.25% |
